# Geocoding Addresses with the U.S. Census Geocoder

**Series:** Geocoding (notebook 1 of 3)
**Time:** about 20 minutes

**In this notebook you will:**
- Learn what geocoding is
- Send one address to the free U.S. Census Geocoder
- Geocode a small list of museum addresses
- Find the addresses that did not match
- Show the results on an interactive map

**You should already know:** how to run a cell in Colab and what a pandas DataFrame is (Foundations notebooks 2a and 3a).

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## What is geocoding?

Geocoding turns a written address into a location on the map.

| Input (text) | Output (coordinates) |
|---|---|
| `1600 Pennsylvania Ave NW, Washington, DC 20500` | latitude 38.8977, longitude -77.0365 |

Most social science data (surveys, business lists, case files) only has addresses.
Geocoding is the step that lets you map and analyze that data spatially.

## Step 1: Import the libraries

In [ ]:
import pandas as pd        # tables
import geopandas as gpd    # tables with a map geometry column
import requests            # sends requests to websites (APIs)

## Step 2: Geocode a single address

The Census Geocoder is a free web service for U.S. addresses. We send it an address and it sends
back a reply in JSON (a nested Python dictionary). The function below sends the request and pulls
out the latitude and longitude. If nothing matches, it returns `None`.

In [ ]:
CENSUS_URL = "https://geocoding.geo.census.gov/geocoder/locations/onelineaddress"

def get_coordinates(address):
    params = {
        "address": address,
        "benchmark": "Public_AR_Current",  # the most recent address database
        "format": "json",
    }
    try:
        reply = requests.get(CENSUS_URL, params=params, timeout=30)
        reply.raise_for_status()
        matches = reply.json()["result"]["addressMatches"]
        if matches:
            point = matches[0]["coordinates"]
            return point["y"], point["x"]   # (latitude, longitude)
    except Exception as error:
        print("Request failed:", error)
    return None, None

In [ ]:
get_coordinates("101 N Grant St, West Lafayette, IN 47906")

You should see two numbers: a latitude near 40.42 and a longitude near -86.91.
That is the Purdue Memorial Union.

> Addresses inside a campus (such as "610 Purdue Mall") are often missing from the Census database
> and return `(None, None)`. Use the street address of a building, as above.

## Step 3: Load a list of addresses

We will geocode 15 well-known U.S. museums. The file is tab separated with two columns: name and address.
`pandas` can read it straight from its web address into memory, so there is nothing to download.

In [ ]:
museums_url = "https://raw.githubusercontent.com/SpatialTurn/DataCollection/main/data/museums.txt"
museums = pd.read_csv(museums_url, sep="\t", header=None, names=["Name", "Address"])
museums.head()

## Step 4: Geocode every row

`apply` runs our function once for each address. This takes about 15 seconds because each
address is a separate request.

In [ ]:
results = museums["Address"].apply(get_coordinates)
museums["Latitude"] = [lat for lat, lon in results]
museums["Longitude"] = [lon for lat, lon in results]

match_rate = museums["Latitude"].notna().mean()
print(f"Matched {match_rate:.0%} of addresses")

## Step 5: Check the addresses that did not match

Geocoders are strict. Two museums usually fail here, for two different reasons:

- **The National Gallery of Art** is written as `Constitution Ave NW`, with **no street number**.
- **The Philadelphia Museum of Art** has a complete address, but it is **not in the Census address
  database**. This happens with parks, campuses and large landmarks.

Always look at the failures.

In [ ]:
museums[museums["Latitude"].isna()]

## Step 6: Turn the table into map points

A **GeoDataFrame** is a DataFrame with a `geometry` column. `EPSG:4326` tells GeoPandas the
numbers are latitude and longitude.

In [ ]:
matched = museums.dropna(subset=["Latitude"])

museum_points = gpd.GeoDataFrame(
    matched,
    geometry=gpd.points_from_xy(matched["Longitude"], matched["Latitude"]),
    crs="EPSG:4326",
)
museum_points.head()

## Step 7: Make an interactive map

In [ ]:
museum_points.explore(tooltip="Name", marker_kwds={"radius": 6}, tiles="Esri.WorldGrayCanvas")

Hover over a point to see its name. You can zoom and pan the map.

> We set `tiles="Esri.WorldGrayCanvas"` for the light grey background map. Some popular backgrounds
> do not work inside Colab: OpenStreetMap shows "403 Access blocked" and CARTO shows "API key required".

## Try it yourself

Geocode your own address (or your office) with `get_coordinates`. Then fix the National Gallery of Art: it sits at the corner of 6th Street and Constitution Avenue NW. The Census Geocoder also understands intersections written as `Street A & Street B`.

*Hint:* the Philadelphia Museum of Art cannot be fixed this way; notebook 01c shows another geocoder that can help.

<details>
<summary><b>Show a solution</b></summary>

```python
get_coordinates("222 W Wood St, West Lafayette, IN 47906")   # replace with your own address

# The National Gallery of Art, written as an intersection
get_coordinates("6th St & Constitution Ave NW, Washington, DC 20565")
```

</details>

In [ ]:
# Write your answer here

## Recap

- Geocoding converts addresses into latitude and longitude.
- The U.S. Census Geocoder is free and works well for complete U.S. street addresses.
- Always check the match rate and look at the addresses that failed.
- `gpd.points_from_xy` turns coordinate columns into map points, and `.explore()` maps them.

**Next:** [Finding Places with OpenStreetMap](https://colab.research.google.com/github/SpatialTurn/DataCollection/blob/main/notebooks/02b_Finding_Places_with_OSM.ipynb)